# Loading TinyDB data

In [1]:
from tinydb import TinyDB
db = TinyDB("store_db.json")
inventory_table = db.table("inventory")
transactions_table = db.table("transactions")

In [ ]:
# Inventory table
import json
print(json.dumps(inventory_table.all(), indent=2))

[
  {
    "item_id": "SG001",
    "name": "Aviator",
    "description": "Originally designed for pilots, these teardrop-shaped lenses with thin metal frames offer timeless appeal. The large lenses provide excellent coverage while the lightweight construction ensures comfort during long wear.",
    "quantity_in_stock": 23,
    "price": 80
  },
  {
    "item_id": "SG002",
    "name": "Wayfarer",
    "description": "Featuring thick, angular frames that make a statement, these sunglasses combine retro charm with modern edge. The rectangular lenses and sturdy acetate construction create a confident look.",
    "quantity_in_stock": 6,
    "price": 95
  },
  {
    "item_id": "SG003",
    "name": "Mystique",
    "description": "Inspired by 1950s glamour, these frames sweep upward at the outer corners to create an elegant, feminine silhouette. The subtle curves and often embellished temples add sophistication to any outfit.",
    "quantity_in_stock": 3,
    "price": 70
  },
  {
    "item_id": "

In [ ]:
# Transaction table
print(json.dumps(transactions_table.all(), indent=2))

[
  {
    "transaction_id": "TXN001",
    "customer_name": "OPENING_BALANCE",
    "transaction_summary": "Daily opening register balance",
    "transaction_amount": 500.0,
    "balance_after_transaction": 500.0,
    "timestamp": "2026-04-19T19:27:43.590775"
  }
]


# Helper functions for TInyDB

In [2]:
# Function to build a readable schema summary for a single table, which helps the LLM to understand the table structure
def build_schema_for_table(tbl, table_name: str, k: int=3) -> str:
    rows = tbl.all()
    
    schema = {}
    for r in rows:
        for k_, v in r.items():
            # Save type of each column
            if k_ not in schema:
                schema[k_] = {
                    "type": type(v).__name__,
                    "examples": []
                }  
            # Store example values
            if len(schema[k_]["examples"]) < k and v not in schema[k_]["examples"]:
                schema[k_]["examples"].append(str(v))
    
    # Build output text
    lines = [f"TABLE: {table_name}", "COLUMNS:"]
    for col, info in schema.items():
        ex = f" | examples: {info['examples']}" if info["examples"] else ""
        lines.append(f"  - {col}: {info['type']}{ex}")
    lines.append(f"ROWS: {len(rows)}")
    lines.append(f"PREVIEW (first 3 rows): {rows}")
    return "\n".join(lines)
    

In [3]:
# Function to combine schemas of inventory and transactions table along with business notes for providing LLM better context
def build_schema_block(inventory_tbl, transactions_tbl) -> str:
    inv = build_schema_for_table(inventory_tbl, "inventory_tbl")
    tx = build_schema_for_table(transactions_tbl, "transactions_tbl")
    
    notes = (
        "NOTES:\n"
        "- inventory_tbl.price is in USD.\n"
        "- inventory_tbl.quantity_in_stock > 0 means available stock.\n"
        "- inventory_tbl.name describes the style (e.g., 'Classic', 'Moon').\n"
        "- transactions_tbl.timestamp is ISO-8601.\n"
    )
    
    return f"{inv}\n\n{tx}\n\n{notes}"

In [4]:
def get_current_balance(transactions_tbl, default: float = 0.0) -> float:
    txns = transactions_tbl.all()
    return txns[-1].get("balance_after_transaction", default) if txns else default

In [5]:
def next_transaction_id(transactions_tbl, prefix: str = "TXN") -> str:
    return f"{prefix}{len(transactions_tbl)+1:03d}"

# Planning with code execution

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api_key)

In [ ]:
# Function to generate plan as code
def generate_llm_code(question: str, inventory_tbl, transactions_tbl):
    schema_block = build_schema_block(inventory_tbl, transactions_tbl)
    prompt = f"""You are a senior data assistant. PLAN BY WRITING PYTHON CODE USING TINYDB.

    Database Schema & Samples (read-only):
    {schema_block}

    Execution Environment (already imported/provided):
    - Variables: db, inventory_tbl, transactions_tbl  # TinyDB Table objects
    - Helpers: get_current_balance(tbl) -> float, next_transaction_id(tbl, prefix="TXN") -> str
    - Natural language: user_request: str  # the original user message

    PLANNING RULES (critical):
    - Derive ALL filters/parameters from user_request (shape/keywords, price ranges "under/over/between", stock mentions,
    quantities, buy/return intent). Do NOT hard-code values.
    - Build TinyDB queries dynamically with Query(). If a constraint isn't in user_request, don't apply it.
    - Be conservative: if intent is ambiguous, do read-only (DRY RUN).

    TRANSACTION POLICY (hard):
    - Do NOT create aggregated multi-item transactions.
    - If the request contains multiple items, create a separate transaction row PER ITEM.
    - For each item:
    - compute its own line total (unit_price * qty),
    - insert ONE transaction with that amount,
    - update balance sequentially (balance += line_total),
    - update the item's stock.
    - If any requested item lacks sufficient stock, do NOT mutate anything; reply with STATUS="insufficient_stock".

    HUMAN RESPONSE REQUIREMENT (hard):
    - You MUST set a variable named `answer_text` (type str) with a short, customer-friendly sentence (1-2 lines).
    - This sentence is the only user-facing message. No dataframes/JSON, no boilerplate disclaimers.
    - If nothing matches, politely say so and offer a nearby alternative (closest style/price) or a next step.

    ACTION POLICY:
    - If the request clearly asks to change state (buy/purchase/return/restock/adjust):
        ACTION="mutate"; SHOULD_MUTATE=True; perform the change and write a matching transaction row.
    Otherwise:
        ACTION="read"; SHOULD_MUTATE=False; simulate and explain briefly as a dry run (in logs only).

    FAILURE & EDGE-CASE HANDLING (must implement):
    - Do not capture outer variables in Query.test. Pass them as explicit args.
    - Always set a short `answer_text`. Also set a string `STATUS` to one of:
    "success", "no_match", "insufficient_stock", "invalid_request", "unsupported_intent".
    - no_match: No items satisfy the filters → suggest the closest in style/price, or invite a different range.
    - insufficient_stock: Item found but stock < requested qty → state available qty and offer the max you can fulfill.
    - invalid_request: Unable to parse essential info (e.g., quantity for a purchase/return) → ask for the missing piece succinctly.
    - unsupported_intent: The action is outside the store's capabilities → provide the nearest supported alternative.
    - In all cases, keep the tone helpful and concise (1-2 sentences). Put technical details (e.g., ACTION/DRY RUN) only in stdout logs.

    OUTPUT CONTRACT:
    - Return ONLY executable Python between these tags (no extra text):
    <execute_python>
    # your python
    </execute_python>

    CODE CHECKLIST (follow in code):
    1) Parse intent & constraints from user_request (regex ok).
    2) Build TinyDB condition incrementally; query inventory_tbl.
    3) If mutate: validate stock, update inventory, insert a transaction (new id, amount, balance, timestamp).
    4) ALWAYS set:
    - `answer_text` (human sentence, required),
    - `STATUS` (see list above).
    Also print a brief log to stdout, e.g., "LOG: ACTION=read DRY_RUN=True STATUS=no_match".
    5) Optional: set `answer_rows` or `answer_json` if useful, but `answer_text` is mandatory.

    TONE EXAMPLES (for `answer_text`):
    - success: "Yes, we have our Classic sunglasses, a round frame, for $60."
    - no_match: "We don't have round frames under $100 in stock right now, but our Moon round frame is available at $120."
    - insufficient_stock: "We only have 1 pair of Classic left; I can reserve that for you."
    - invalid_request: "I can help with that—how many pairs would you like to purchase?"
    - unsupported_intent: "We can't refurbish frames, but I can suggest similar new models."

    Constraints:
    - Use TinyDB Query for filtering. Standard library imports only if needed.
    - Keep code clear and commented with numbered steps.

    User request:
    {question}
    """

    res = client.chat.completions.create(
        model="o4-mini",
        temperature=1.0,
        messages=[
            {
                "role":"system",
                "content":"You write safe, well-commented TinyDB code to handle data questions and updates."
            },
            {
                "role":"user",
                "content":prompt
            }
        ]
    )
    
    content = res.choices[0].message.content
    return content

In [32]:
content = generate_llm_code(
    question="Do you have any round sunglasses in stock that are under $100?",
    inventory_tbl=inventory_table,
    transactions_tbl=transactions_table
)

print(content)

<execute_python>
# 1) Parse constraints from user_request
user_text = user_request.lower()
import re

# Shape constraint: look for 'round'
shape_keyword = None
if re.search(r'\bround\b', user_text):
    shape_keyword = 'round'

# Price constraint: under $X or below $X
price_max = None
m = re.search(r'under\s*\$?(\d+)', user_text)
if m:
    price_max = float(m.group(1))

# 2) Build TinyDB query incrementally
from tinydb import Query
Inventory = Query()
query = (Inventory.quantity_in_stock > 0)
if price_max is not None:
    query &= (Inventory.price < price_max)
if shape_keyword:
    # check description for shape keyword, case-insensitive
    query &= Inventory.description.test(lambda desc, kw=shape_keyword: kw in desc.lower())

# 3) Execute search (read-only)
results = inventory_tbl.search(query)

# 4) Formulate response
if results:
    # use the first matching item
    item = results[0]
    STATUS = "success"
    answer_text = f"Yes, we have our {item['name']} sunglasses, a round frame

In [ ]:
import re

# Function to extract code
def extract_code(text: str) -> str:
    match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", text)
    code = match.group(1).strip()
    return code

In [ ]:
from typing import Any, Dict, Optional
import io
import sys
import traceback
from tinydb import Query, where

# Function to execute code
def execute_code(content: str, db, inventory_tbl, transactions_tbl, user_request: Optional[str] = None) -> Dict[str, Any]:
    code = extract_code(content)
    
    SAFE_GLOBALS = {
        "Query": Query,
        "get_current_balance": get_current_balance,
        "next_transaction_id": next_transaction_id,
        "user_request": user_request or "",
    }
    SAFE_LOCALS = {
        "db": db,
        "inventory_tbl": inventory_tbl,
        "transactions_tbl": transactions_tbl,
    }
    
    # To captute print() output
    output_buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = output_buffer
    
    error = None
    
    try:
        # Run generated code
        exec(code, SAFE_GLOBALS, SAFE_LOCALS)
    except Exception:
        # Save error messages if any
        error = traceback.format_exc()
    finally:
        sys.stdout = old_stdout
    
    printed_output = output_buffer.getvalue().strip()
    
    # Get answer created by the generated code
    answer = (
        SAFE_LOCALS.get("answer_text")
        or SAFE_LOCALS.get("answer_rows")
        or SAFE_LOCALS.get("answer_json")
    )
    
    return{
        "code": code,
        "stdout": printed_output,
        "answer": answer,
        "error": error,
        "inventory_tbl": inventory_tbl.all(),
        "transactions_tbl": transactions_tbl.all(),
    }
    

In [33]:
result = execute_code(
    content, 
    db=db, 
    inventory_tbl=inventory_table, 
    transactions_tbl=transactions_table
)

print(result["answer"])

Yes, we have our Aviator sunglasses, a round frame, for $80.


In [34]:
content = generate_llm_code(
    question="Return 2 Aviator sunglasses I bought last week.",
    inventory_tbl=inventory_table,
    transactions_tbl=transactions_table
)

print(content)

<execute_python>
# 1) Parse intent & constraints from user_request
import re
from tinydb import Query
import datetime

# Initialize status and answer_text
STATUS = None
answer_text = ""

# Attempt to extract quantity
qty_match = re.search(r'\b(\d+)\b', user_request)
if not qty_match:
    # Missing quantity
    STATUS = "invalid_request"
    answer_text = "I can help with that—how many pairs would you like to return?"
    print(f"LOG: ACTION=mutate DRY_RUN=False STATUS={STATUS}")
else:
    qty = int(qty_match.group(1))
    # Attempt to extract item style name (before "sunglasses" or end of string)
    name_match = re.search(r'\d+\s+([\w\s]+?)(?:\s+sunglasses|\s*$)', user_request, re.IGNORECASE)
    if not name_match:
        # Missing style
        STATUS = "invalid_request"
        answer_text = "Which style would you like to return?"
        print(f"LOG: ACTION=mutate DRY_RUN=False STATUS={STATUS}")
    else:
        item_name = name_match.group(1).strip()
        # 2) Query inventory

In [36]:
result = execute_code(
    content, 
    db=db, 
    inventory_tbl=inventory_table, 
    transactions_tbl=transactions_table
)

print(result["answer"])

I can help with that—how many pairs would you like to return?


# Building the full Pipeline

In [40]:
def customer_service_agent(question: str, db, inventory_tbl, transactions_tbl) -> dict:
    # 1. Show the question
    print("User Question :- ")
    print(question)
    
    # 2. Generated plan-as-code 
    plan_as_code = generate_llm_code(
        question,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl
    )
    print("Generated code :- ")
    print(plan_as_code)
    
    # 3. Execute
    result = execute_code(
        plan_as_code,
        db=db,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        user_request=question
    )
    print("Answer :-")
    print(result["answer"])
    
    # 4. return outputs
    return {
        "full_content": plan_as_code,
        "exec":{
            "code": result["code"],
            "stdout": result["stdout"],
            "error": result["error"],
            "answer": result["answer"],
        }
    }
    

In [41]:
out = customer_service_agent(
    question="I want to buy 3 pairs of classic sunglasses and 1 pair of aviator sunglasses.",
    db=db,
    inventory_tbl=inventory_table,
    transactions_tbl=transactions_table
)

User Question :- 
I want to buy 3 pairs of classic sunglasses and 1 pair of aviator sunglasses.
Generated code :- 
<execute_python>
import re
from tinydb import Query
from datetime import datetime

# 1) Initialize STATUS and answer_text
STATUS = None
answer_text = ""

# 2) Parse purchase intent and extract item names and quantities
pattern = r'(\d+)\s+pairs? of\s+([A-Za-z]+)\s+sunglasses'
matches = re.findall(pattern, user_request, re.IGNORECASE)
items = [(int(q), name.capitalize()) for q, name in matches]

if not items:
    # Missing quantity/item info
    answer_text = "I can help with that—how many pairs would you like to purchase?"
    STATUS = "invalid_request"
    print(f"LOG: ACTION=read DRY_RUN=True STATUS={STATUS}")
else:
    # 3) Validate availability for each requested item
    insufficient = False
    no_matches = []
    insuff_detail = None
    processed = []
    for qty, name in items:
        inv_Q = Query()
        results = inventory_tbl.search(inv_Q.name.test(lambda x